# ==========================================================
# Title: Google Colab Working Project – Customer Support RAG Assistant
# Name: Syad Ali
# Intern ID: IN126055102
# Date: 22 April 2026
# ==========================================================

In [1]:
!pip install -q langchain langchain-community langgraph chromadb sentence-transformers pypdf huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

In [2]:
!pip install -q langchain-text-splitters

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFaceHub

from langgraph.graph import StateGraph
from typing import TypedDict

import os

In [4]:
from google.colab import files
uploaded = files.upload()

Saving College_Student_Support_Handbook1.pdf to College_Student_Support_Handbook1.pdf


In [5]:
loader = PyPDFLoader("College_Student_Support_Handbook1.pdf")
documents = loader.load()

print("Total pages loaded:", len(documents))

Total pages loaded: 2


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 6


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_3133/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

In [9]:
import getpass
os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your HuggingFace API Key: ")

Enter your HuggingFace API Key: ··········


In [13]:
!pip install -q transformers accelerate

In [12]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 61.3 MB/s eta 0:00:00


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [14]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_length=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [15]:
def rag_pipeline(query):
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using ONLY the context below.
Give a SHORT and DIRECT answer.

Context:
{context}

Question: {query}

Answer:
"""

    response = generate_answer(prompt)

    return response.strip()

In [16]:
query = "What is the semester registration policy??"
print(rag_pipeline(query))

Students must complete semester registration before the announced deadline.


In [19]:
print(rag_pipeline("What happens if semester registration is not completed?"))
print(rag_pipeline("What is the classroom discipline policy?"))
print(rag_pipeline("What happens if assignments are submitted late?"))
print(rag_pipeline("Is internship mandatory for students?"))
print(rag_pipeline("What is the digital learning policy?"))

unregistered students may not be allowed to attend classes
Students must maintain silence and discipline inside classrooms.
reduced marks
Yes
Students should regularly check online portals for notices, study materials, and announcements.


In [20]:
from typing import TypedDict

class GraphState(TypedDict):
    query: str
    context: str
    response: str
    confidence: float
    route: str

In [21]:
def process_node(state):
    query = state["query"]

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using ONLY the context below.
Give a SHORT and DIRECT answer.

Context:
{context}

Question: {query}

Answer:
"""

    response = generate_answer(prompt)

    # simple confidence logic
    confidence = 0.9 if len(context) > 100 else 0.3

    return {
        "query": query,
        "context": context,
        "response": response,
        "confidence": confidence
    }

In [22]:
def route_node(state):
    if state["confidence"] < 0.5:
        return "hitl"
    else:
        return "output"

In [23]:
def hitl_node(state):
    print("⚠️ Escalating to Human...")

    # simulate human answer
    human_response = "Please contact the administration for more details."

    return {
        "query": state["query"],
        "context": state["context"],
        "response": human_response,
        "confidence": 1.0,
        "route": "hitl"
    }

In [24]:
def output_node(state):
    return state

In [25]:
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

# nodes
builder.add_node("process", process_node)
builder.add_node("hitl", hitl_node)
builder.add_node("output", output_node)

# edges
builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    route_node,
    {
        "hitl": "hitl",
        "output": "output"
    }
)

builder.add_edge("hitl", "output")

graph = builder.compile()

In [26]:
result = graph.invoke({
    "query": "What is the semester registration policy?"
})

print("Final Answer:", result["response"])
print("Route:", result.get("route", "auto"))

Final Answer: Students must complete semester registration before the announced deadline.
Route: auto


In [27]:
test_queries = [
    # ✅ Normal queries (should go AUTO)
    "What is the semester registration policy?",
    "What is the classroom discipline policy?",
    "What is the assignment submission policy?",
    "What is the internal assessment policy?",
    "Is internship mandatory for students?",

    # ⚠️ Medium queries
    "What happens if assignments are submitted late?",
    "Which email should students use for communication?",
    "Where should students park their vehicles?",
    "What should I do if I lose an item?",

    # 🚨 Should trigger HITL (unknown / complex)
    "Can I get hostel room immediately?",
    "What is today's exam timetable?",
    "Can you increase my marks?",
    "Is there any special permission for attendance shortage?"
]

for q in test_queries:
    print("\n" + "="*50)
    print("Query:", q)

    result = graph.invoke({"query": q})

    print("Answer:", result["response"])
    print("Route:", result.get("route", "auto"))


Query: What is the semester registration policy?
Answer: Students must complete semester registration before the announced deadline.
Route: auto

Query: What is the classroom discipline policy?
Answer: Students must maintain silence and discipline inside classrooms.
Route: auto

Query: What is the assignment submission policy?
Answer: Assignments must be submitted on or before the due date. Late submissions may receive reduced marks.
Route: auto

Query: What is the internal assessment policy?
Answer: Students must attend quizzes, seminars, and internal tests as scheduled.
Route: auto

Query: Is internship mandatory for students?
Answer: Yes
Route: auto

Query: What happens if assignments are submitted late?
Answer: reduced marks
Route: auto

Query: Which email should students use for communication?
Answer: official college email IDs
Route: auto

Query: Where should students park their vehicles?
Answer: designated parking areas
Route: auto

Query: What should I do if I lose an item?
An

In [29]:
test_queries = [
    # AUTO Queries
    "What is the semester registration policy?",
    "What is the assignment submission policy?",
    "Is internship mandatory for students?",

    # HITL Queries
    "Can you increase my marks?",
    "What is today's exam timetable?",
    "Can I get hostel room immediately?",
    "Please give me special permission for attendance shortage.",
    "Can you change my internal marks?"
]

for q in test_queries:
    print("\n" + "="*50)
    print("Query:", q)

    result = graph.invoke({"query": q})

    print("Answer:", result["response"])
    print("Route:", result.get("route", "auto"))


Query: What is the semester registration policy?
Answer: Students must complete semester registration before the announced deadline.
Route: auto

Query: What is the assignment submission policy?
Answer: Assignments must be submitted on or before the due date. Late submissions may receive reduced marks.
Route: auto

Query: Is internship mandatory for students?
Answer: Yes
Route: auto

Query: Can you increase my marks?
Answer: 0
Route: auto

Query: What is today's exam timetable?
Answer: 8:00 AM
Route: auto

Query: Can I get hostel room immediately?
Answer: DIRECT
Route: auto

Query: Please give me special permission for attendance shortage.
Answer: DIRECT
Route: auto

Query: Can you change my internal marks?
Answer: 0
Route: auto
